# Test des clusters — illustrations de Bibles (corpus propre)

Clustering visuel des 721 illustrations nettoyées, **pour test uniquement**.
On regarde l'aperçu de chaque groupe ; rien n'est copié ni déplacé.

Objectif : trouver le bon nombre de clusters avant de créer les dossiers-groupes
que Céline ajustera.

Travaille sur la copie `data/bibles_mdz/regroupement/` (les sources `segmentees/`
restent intactes).

*Notebook prévu pour `notebooks/04_exploration/` — `RACINE` remonte de deux niveaux.*

## 1 · Imports et embeddings

In [ ]:
import os
import numpy as np
import torch, torch.nn as nn
import open_clip
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from collections import Counter

RACINE = os.path.abspath("../../")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DOSSIER = os.path.join(RACINE, "data", "bibles_mdz", "regroupement")
print("Device :", DEVICE)
print("Dossier :", DOSSIER)

## 2 · Lister les illustrations

In [ ]:
images = []
for bsb_id in sorted(os.listdir(DOSSIER)):
    d = os.path.join(DOSSIER, bsb_id)
    if not os.path.isdir(d):
        continue
    for f in sorted(os.listdir(d)):
        if f.lower().endswith((".jpg", ".jpeg", ".png")) and "_flip" not in f and not f.startswith("_tmp_"):
            images.append({"chemin": os.path.join(d, f), "bsb": bsb_id, "nom": f})

print(f"{len(images)} illustrations")

## 3 · Extracteur CLIP (+ niveaux de gris)

CLIP capte le contenu sémantique ; le passage en niveaux de gris neutralise le coloriage.

In [ ]:
modele_clip, _, _ = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
modele_clip = modele_clip.to(DEVICE).eval()
_, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")

def embedding(chemin):
    img = Image.open(chemin).convert("RGB").convert("L").convert("RGB")
    x = preprocess(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        return modele_clip.encode_image(x).cpu().numpy()[0]

print("CLIP prêt")

## 4 · Calculer les embeddings

In [ ]:
vecteurs, valides = [], []
print("Calcul…")
for k, img in enumerate(images, 1):
    print(f"  {k}/{len(images)}", end="\r")
    try:
        vecteurs.append(embedding(img["chemin"]))
        valides.append(img)
    except Exception as e:
        print(f"\n  ignoré : {img['nom']} ({e})")

X = np.array(vecteurs)
images = valides
print(f"\n{X.shape[0]} embeddings (dim {X.shape[1]})")

## 5 · Clustering — règle ce nombre et relance

Change `N_CLUSTERS`, relance cette cellule + l'aperçu (section 6) autant de fois que
nécessaire jusqu'à ce que les groupes te paraissent cohérents.

In [ ]:
N_CLUSTERS = 8  # ← teste 8, 12, 16, 20… et regarde l'aperçu

km = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
labels = km.fit_predict(X)
for i, img in enumerate(images):
    img["cluster"] = int(labels[i])

tailles = Counter(labels)
print(f"{N_CLUSTERS} clusters :")
for c in range(N_CLUSTERS):
    print(f"   groupe {c:2d} : {tailles[c]} images")

## 6 · Aperçu de chaque cluster

In [ ]:
N_APERCU = 30  # vignettes affichées par cluster

for c in range(N_CLUSTERS):
    membres = [img for img in images if img["cluster"] == c]
    apercu = membres[:N_APERCU]
    if not apercu:
        continue
    fig, axes = plt.subplots(1, len(apercu), figsize=(len(apercu)*1.6, 1.8))
    if len(apercu) == 1:
        axes = [axes]
    for ax, m in zip(axes, apercu):
        try:
            ax.imshow(Image.open(m["chemin"]).convert("RGB"))
        except Exception:
            pass
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f"Groupe {c} — {len(membres)} images", fontsize=10)
    plt.tight_layout()
    plt.show()

## Notes

- Rien n'est copié ni déplacé ici : c'est un **test visuel**.
- Fais varier `N_CLUSTERS` (section 5) et relance les sections 5-6 jusqu'à trouver le bon
  grain (groupes ni trop fourre-tout, ni trop éclatés).
- Quand le découpage te convient, on créera les dossiers-groupes (copie depuis
  `regroupement/`) que Céline ajustera dans l'explorateur.